# Test du modèle en Retrieval

Ce notebook teste votre modèle LoRA fine-tuné en **retrieval** :
- Étant donné un **code**, retrouver le **feedback** correspondant
- Étant donné un **feedback**, retrouver le **code** correspondant

Cela vous permettra de voir concrètement où le modèle échoue.

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoModel, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, average_precision_score

sns.set_style('whitegrid')
%matplotlib inline

## 1. Chargement du modèle

In [ ]:
# Configuration
BASE_MODEL = "Salesforce/SFR-Embedding-Code-400M_R"
LORA_CHECKPOINT = "../checkpoints/lora_contrastive_fixed/best_model"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")
print(f"Loading base model: {BASE_MODEL}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model
base_model = AutoModel.from_pretrained(BASE_MODEL)

# Load LoRA weights
model = PeftModel.from_pretrained(base_model, LORA_CHECKPOINT)
model = model.to(DEVICE)
model.eval()

print("✓ Model loaded successfully")

## 2. Fonction d'encoding

In [ ]:
def mean_pooling(token_embeddings, attention_mask):
    """Mean pooling with attention mask."""
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


@torch.no_grad()
def encode_texts(texts, max_length=512, batch_size=32, show_progress=True):
    """Encode a list of texts into embeddings."""
    all_embeddings = []
    
    iterator = range(0, len(texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Encoding")
    
    for i in iterator:
        batch_texts = texts[i:i+batch_size]
        
        # Tokenize
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(DEVICE)
        
        # Encode
        outputs = model(**inputs)
        embeddings = mean_pooling(outputs.last_hidden_state, inputs['attention_mask'])
        embeddings = F.normalize(embeddings, p=2, dim=1)
        
        all_embeddings.append(embeddings.cpu())
    
    return torch.cat(all_embeddings, dim=0)


print("✓ Encoding functions defined")

## 3. Chargement des données de test

In [ ]:
# Load dataset (same splits as training)
print("Loading dataset...")
hf_dataset = load_dataset("matis35/RAFT")
all_dfs = [split_data.to_pandas() for split_data in hf_dataset.values()]
df_feedback = pd.concat(all_dfs, ignore_index=True)

df_clustered = pd.read_csv("../data/dataset_clustered_no_tests.csv")
df_clustered = df_clustered.rename(columns={'code_snippet': 'code'})

df = df_feedback.merge(
    df_clustered[['code', 'cluster_kmeans']],
    on='code',
    how='inner'
)
df['cluster_kmeans'] = df['cluster_kmeans'].fillna(-1).astype(int)

# Same split as training
cluster_counts = df['cluster_kmeans'].value_counts()
min_cluster_size = cluster_counts.min()

if min_cluster_size >= 3:
    train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df['cluster_kmeans'])
    train_df, val_df = train_test_split(train_df, test_size=0.111, random_state=42, stratify=train_df['cluster_kmeans'])
else:
    train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
    train_df, val_df = train_test_split(train_df, test_size=0.111, random_state=42)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# Take a subset for faster testing (optional)
USE_SUBSET = True
SUBSET_SIZE = 200

if USE_SUBSET:
    val_df_subset = val_df.sample(n=min(SUBSET_SIZE, len(val_df)), random_state=42)
    test_df_subset = test_df.sample(n=min(SUBSET_SIZE, len(test_df)), random_state=42)
    print(f"\nUsing subsets: Val={len(val_df_subset)}, Test={len(test_df_subset)}")
else:
    val_df_subset = val_df
    test_df_subset = test_df

## 4. Encoding des données

In [ ]:
print("Encoding validation set...")
val_codes = val_df_subset['code'].tolist()
val_feedbacks = val_df_subset['feedback'].tolist()

val_code_embeddings = encode_texts(val_codes, max_length=512, batch_size=32)
val_feedback_embeddings = encode_texts(val_feedbacks, max_length=256, batch_size=32)

print("\nEncoding test set...")
test_codes = test_df_subset['code'].tolist()
test_feedbacks = test_df_subset['feedback'].tolist()

test_code_embeddings = encode_texts(test_codes, max_length=512, batch_size=32)
test_feedback_embeddings = encode_texts(test_feedbacks, max_length=256, batch_size=32)

print("\n✓ All embeddings computed")
print(f"Val code embeddings: {val_code_embeddings.shape}")
print(f"Test code embeddings: {test_code_embeddings.shape}")

## 5. Retrieval Metrics

**Task** : Étant donné un code, retrouver le feedback correct parmi tous les feedbacks

In [ ]:
def compute_retrieval_metrics(query_embeddings, key_embeddings, top_k=[1, 5, 10]):
    """
    Compute retrieval metrics.
    
    Args:
        query_embeddings: [N, D] tensor of query embeddings
        key_embeddings: [N, D] tensor of key embeddings
        
    Returns:
        Dictionary with Recall@K and MRR
    """
    # Compute similarity matrix
    similarity = torch.matmul(query_embeddings, key_embeddings.T)  # [N, N]
    
    # Get rankings
    _, indices = similarity.topk(k=max(top_k), dim=1)  # [N, K]
    
    # Ground truth: each query corresponds to the key at the same index
    correct_indices = torch.arange(len(query_embeddings))
    
    metrics = {}
    
    # Recall@K
    for k in top_k:
        top_k_indices = indices[:, :k]
        recall_at_k = (top_k_indices == correct_indices.unsqueeze(1)).any(dim=1).float().mean().item()
        metrics[f'Recall@{k}'] = recall_at_k
    
    # Mean Reciprocal Rank (MRR)
    ranks = (indices == correct_indices.unsqueeze(1)).nonzero(as_tuple=True)[1] + 1
    mrr = (1.0 / ranks.float()).mean().item()
    metrics['MRR'] = mrr
    
    # Mean rank
    metrics['Mean_Rank'] = ranks.float().mean().item()
    
    return metrics, similarity


print("✓ Metrics function defined")

## 6. Évaluation sur Validation Set

In [ ]:
print("="*80)
print("VALIDATION SET RETRIEVAL")
print("="*80)

# Code → Feedback retrieval
print("\n1. Code → Feedback (given code, find feedback)")
val_c2f_metrics, val_c2f_sim = compute_retrieval_metrics(
    val_code_embeddings,
    val_feedback_embeddings,
    top_k=[1, 5, 10, 20]
)

for metric, value in val_c2f_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Feedback → Code retrieval
print("\n2. Feedback → Code (given feedback, find code)")
val_f2c_metrics, val_f2c_sim = compute_retrieval_metrics(
    val_feedback_embeddings,
    val_code_embeddings,
    top_k=[1, 5, 10, 20]
)

for metric, value in val_f2c_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 7. Évaluation sur Test Set

In [ ]:
print("\n" + "="*80)
print("TEST SET RETRIEVAL")
print("="*80)

# Code → Feedback retrieval
print("\n1. Code → Feedback (given code, find feedback)")
test_c2f_metrics, test_c2f_sim = compute_retrieval_metrics(
    test_code_embeddings,
    test_feedback_embeddings,
    top_k=[1, 5, 10, 20]
)

for metric, value in test_c2f_metrics.items():
    print(f"  {metric}: {value:.4f}")

# Feedback → Code retrieval
print("\n2. Feedback → Code (given feedback, find code)")
test_f2c_metrics, test_f2c_sim = compute_retrieval_metrics(
    test_feedback_embeddings,
    test_code_embeddings,
    top_k=[1, 5, 10, 20]
)

for metric, value in test_f2c_metrics.items():
    print(f"  {metric}: {value:.4f}")

## 8. Comparaison Val vs Test

In [ ]:
# Create comparison dataframe
comparison_data = []

for metric in ['Recall@1', 'Recall@5', 'Recall@10', 'MRR', 'Mean_Rank']:
    comparison_data.append({
        'Metric': metric,
        'Val (Code→Feedback)': val_c2f_metrics[metric],
        'Test (Code→Feedback)': test_c2f_metrics[metric],
        'Gap (%)': (test_c2f_metrics[metric] - val_c2f_metrics[metric]) / val_c2f_metrics[metric] * 100
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("VAL vs TEST COMPARISON (Code→Feedback)")
print("="*80)
print(comparison_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Recall comparison
recall_metrics = ['Recall@1', 'Recall@5', 'Recall@10']
val_recalls = [val_c2f_metrics[m] for m in recall_metrics]
test_recalls = [test_c2f_metrics[m] for m in recall_metrics]

x = np.arange(len(recall_metrics))
width = 0.35

axes[0].bar(x - width/2, val_recalls, width, label='Validation', alpha=0.8)
axes[0].bar(x + width/2, test_recalls, width, label='Test', alpha=0.8)
axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Recall Metrics: Val vs Test')
axes[0].set_xticks(x)
axes[0].set_xticklabels(recall_metrics)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MRR and Mean Rank
other_metrics = ['MRR', 'Mean_Rank']
val_other = [val_c2f_metrics[m] for m in other_metrics]
test_other = [test_c2f_metrics[m] for m in other_metrics]

x2 = np.arange(len(other_metrics))
axes[1].bar(x2 - width/2, val_other, width, label='Validation', alpha=0.8)
axes[1].bar(x2 + width/2, test_other, width, label='Test', alpha=0.8)
axes[1].set_xlabel('Metric')
axes[1].set_ylabel('Score')
axes[1].set_title('MRR and Mean Rank: Val vs Test')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(other_metrics)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/retrieval_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved to ../results/retrieval_comparison.png")

## 9. Analyse des embeddings

Visualisons la distribution des similarités

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Val: positive vs negative similarities
val_positive_sim = torch.diagonal(val_c2f_sim).numpy()
val_negative_sim = val_c2f_sim[~torch.eye(len(val_c2f_sim), dtype=bool)].numpy()

axes[0, 0].hist(val_positive_sim, bins=50, alpha=0.7, label='Positive pairs', color='green')
axes[0, 0].hist(val_negative_sim, bins=50, alpha=0.7, label='Negative pairs', color='red')
axes[0, 0].set_xlabel('Cosine Similarity')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('VAL: Similarity Distribution')
axes[0, 0].legend()
axes[0, 0].axvline(val_positive_sim.mean(), color='green', linestyle='--', label=f'Mean pos: {val_positive_sim.mean():.3f}')
axes[0, 0].axvline(val_negative_sim.mean(), color='red', linestyle='--', label=f'Mean neg: {val_negative_sim.mean():.3f}')

# Test: positive vs negative similarities
test_positive_sim = torch.diagonal(test_c2f_sim).numpy()
test_negative_sim = test_c2f_sim[~torch.eye(len(test_c2f_sim), dtype=bool)].numpy()

axes[0, 1].hist(test_positive_sim, bins=50, alpha=0.7, label='Positive pairs', color='green')
axes[0, 1].hist(test_negative_sim, bins=50, alpha=0.7, label='Negative pairs', color='red')
axes[0, 1].set_xlabel('Cosine Similarity')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('TEST: Similarity Distribution')
axes[0, 1].legend()
axes[0, 1].axvline(test_positive_sim.mean(), color='green', linestyle='--')
axes[0, 1].axvline(test_negative_sim.mean(), color='red', linestyle='--')

# Embedding variance
val_code_var = val_code_embeddings.var(dim=0).mean().item()
val_feedback_var = val_feedback_embeddings.var(dim=0).mean().item()
test_code_var = test_code_embeddings.var(dim=0).mean().item()
test_feedback_var = test_feedback_embeddings.var(dim=0).mean().item()

variance_data = {
    'Split': ['Val', 'Val', 'Test', 'Test'],
    'Type': ['Code', 'Feedback', 'Code', 'Feedback'],
    'Variance': [val_code_var, val_feedback_var, test_code_var, test_feedback_var]
}
variance_df = pd.DataFrame(variance_data)

x_pos = np.arange(len(variance_df))
colors = ['blue', 'orange', 'blue', 'orange']
axes[1, 0].bar(x_pos, variance_df['Variance'], color=colors, alpha=0.7)
axes[1, 0].set_xlabel('Split - Type')
axes[1, 0].set_ylabel('Variance')
axes[1, 0].set_title('Embedding Variance')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels([f"{row['Split']}\n{row['Type']}" for _, row in variance_df.iterrows()])
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Add values on bars
for i, v in enumerate(variance_df['Variance']):
    axes[1, 0].text(i, v + 0.00005, f'{v:.5f}', ha='center', va='bottom', fontsize=9)

# Margin distribution
margin_data = {
    'Split': ['Val', 'Test'],
    'Margin': [
        val_positive_sim.mean() - val_negative_sim.mean(),
        test_positive_sim.mean() - test_negative_sim.mean()
    ]
}
margin_df = pd.DataFrame(margin_data)

axes[1, 1].bar(margin_df['Split'], margin_df['Margin'], color=['green', 'red'], alpha=0.7)
axes[1, 1].set_xlabel('Split')
axes[1, 1].set_ylabel('Margin (Pos - Neg)')
axes[1, 1].set_title('Similarity Margin')
axes[1, 1].grid(True, alpha=0.3, axis='y')

for i, row in margin_df.iterrows():
    axes[1, 1].text(i, row['Margin'] + 0.002, f"{row['Margin']:.4f}", ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../results/embedding_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved to ../results/embedding_analysis.png")

## 10. Exemples de retrieval réussis et ratés

In [ ]:
def show_retrieval_examples(split_name, code_embeddings, feedback_embeddings, codes, feedbacks, n_examples=3):
    """
    Show examples of successful and failed retrievals.
    """
    similarity = torch.matmul(code_embeddings, feedback_embeddings.T)
    top_5_indices = similarity.topk(k=5, dim=1)[1]  # [N, 5]
    
    # Find successful retrievals (correct in top-1)
    correct_mask = (top_5_indices[:, 0] == torch.arange(len(codes)))
    successful_indices = correct_mask.nonzero(as_tuple=True)[0][:n_examples]
    
    # Find failed retrievals (correct NOT in top-5)
    failed_mask = ~(top_5_indices == torch.arange(len(codes)).unsqueeze(1)).any(dim=1)
    failed_indices = failed_mask.nonzero(as_tuple=True)[0][:n_examples]
    
    print("\n" + "="*100)
    print(f"{split_name} - SUCCESSFUL RETRIEVALS (Top-1 correct)")
    print("="*100)
    
    for i, idx in enumerate(successful_indices, 1):
        idx = idx.item()
        print(f"\n### Example {i}")
        print(f"\n**Query Code:**\n{codes[idx][:200]}...")
        print(f"\n**Correct Feedback:**\n{feedbacks[idx][:200]}...")
        print(f"\n**Similarity:** {similarity[idx, idx]:.4f}")
    
    print("\n" + "="*100)
    print(f"{split_name} - FAILED RETRIEVALS (Correct not in Top-5)")
    print("="*100)
    
    for i, idx in enumerate(failed_indices, 1):
        idx = idx.item()
        retrieved_idx = top_5_indices[idx, 0].item()
        
        print(f"\n### Example {i}")
        print(f"\n**Query Code:**\n{codes[idx][:200]}...")
        print(f"\n**Correct Feedback (rank > 5):**\n{feedbacks[idx][:200]}...")
        print(f"**Similarity:** {similarity[idx, idx]:.4f}")
        print(f"\n**Retrieved Feedback (rank 1):**\n{feedbacks[retrieved_idx][:200]}...")
        print(f"**Similarity:** {similarity[idx, retrieved_idx]:.4f}")


# Show examples for both splits
show_retrieval_examples("VALIDATION", val_code_embeddings, val_feedback_embeddings, val_codes, val_feedbacks)
show_retrieval_examples("TEST", test_code_embeddings, test_feedback_embeddings, test_codes, test_feedbacks)

## 11. Résumé final

In [ ]:
print("\n" + "="*100)
print("SUMMARY")
print("="*100)

print("\n### Retrieval Performance (Code → Feedback)")
print(f"\nValidation:")
print(f"  Recall@1:  {val_c2f_metrics['Recall@1']:.2%}")
print(f"  Recall@10: {val_c2f_metrics['Recall@10']:.2%}")
print(f"  MRR:       {val_c2f_metrics['MRR']:.4f}")

print(f"\nTest:")
print(f"  Recall@1:  {test_c2f_metrics['Recall@1']:.2%}")
print(f"  Recall@10: {test_c2f_metrics['Recall@10']:.2%}")
print(f"  MRR:       {test_c2f_metrics['MRR']:.4f}")

print(f"\nGap (Test vs Val):")
gap_recall1 = (test_c2f_metrics['Recall@1'] - val_c2f_metrics['Recall@1']) / val_c2f_metrics['Recall@1'] * 100
gap_recall10 = (test_c2f_metrics['Recall@10'] - val_c2f_metrics['Recall@10']) / val_c2f_metrics['Recall@10'] * 100
gap_mrr = (test_c2f_metrics['MRR'] - val_c2f_metrics['MRR']) / val_c2f_metrics['MRR'] * 100

print(f"  Recall@1:  {gap_recall1:+.1f}%")
print(f"  Recall@10: {gap_recall10:+.1f}%")
print(f"  MRR:       {gap_mrr:+.1f}%")

print("\n### Embedding Quality")
print(f"\nValidation:")
print(f"  Code variance:     {val_code_var:.6f}")
print(f"  Feedback variance: {val_feedback_var:.6f}")
print(f"  Margin:            {val_positive_sim.mean() - val_negative_sim.mean():.4f}")

print(f"\nTest:")
print(f"  Code variance:     {test_code_var:.6f}")
print(f"  Feedback variance: {test_feedback_var:.6f}")
print(f"  Margin:            {test_positive_sim.mean() - test_negative_sim.mean():.4f}")

print("\n" + "="*100)

# Diagnostic
if gap_recall1 < -30:
    print("\n⚠️  DIAGNOSTIC: Major generalization gap detected!")
    print("   The model performs significantly worse on test set.")
    print("   Likely causes:")
    print("   - Overfitting to training examples")
    print("   - Insufficient model capacity")
    print("   - Need more regularization or data augmentation")
elif gap_recall1 < -10:
    print("\n⚠️  DIAGNOSTIC: Moderate generalization gap.")
    print("   Some overfitting present but not critical.")
else:
    print("\n✓ DIAGNOSTIC: Good generalization!")
    print("  Model performs consistently across val and test sets.")